# RF-DETR Cross-Dataset Evaluation — Model Comparison

Comparing two models:
- **arcade_dataset2_trainval** — trained on combined ARCADE + dataset2 (train+val)
- **dataset2_augs** — trained on dataset2_split only

Evaluated on:
1. **dataset2_split/test** — 1252 images (Russian dataset, patient-split)
2. **cadica_50plus/train** — 1465 images (CADICA ≥50% stenosis, used as test)
3. **stenosis_arcade_singlelabel/test** — 300 images (ARCADE, single-class labels)

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from collections import defaultdict

ROOT = Path('/home/dsa/stenosis')
EVAL_DIR_A = ROOT / 'rfdetr_runs' / 'arcade_dataset2_trainval'
EVAL_DIR_B = ROOT / 'rfdetr_runs' / 'dataset2_augs'
CONF = 0.15

MODEL_A = 'arcade_dataset2_trainval'
MODEL_B = 'dataset2_augs'

## 1. Summary Metrics Table

In [ ]:
import pandas as pd

with open(EVAL_DIR_A / 'eval_summary.json') as f:
    summary_a = json.load(f)
with open(EVAL_DIR_B / 'eval_summary.json') as f:
    summary_b = json.load(f)

rows = []
for name in summary_a:
    for model_name, s in [(MODEL_A, summary_a), (MODEL_B, summary_b)]:
        r = s[name]
        rows.append({
            'Dataset': name,
            'Model': model_name,
            'Images': r['num_images'],
            'mAP@0.25': r['mAP@0.25'],
            'mAP@0.50': r['mAP@0.5'],
            'mAR@0.25': r['mAR@0.25'],
            'mAR@0.50': r['mAR@0.5'],
            f'Recall@IoU0.25': r[f'recall@IoU0.25_conf{CONF}'],
            f'Recall@IoU0.3': r[f'recall@IoU0.3_conf{CONF}'],
            f'Recall@IoU0.5': r[f'recall@IoU0.5_conf{CONF}'],
            f'Prec@IoU0.25': r[f'precision@IoU0.25_conf{CONF}'],
            f'Prec@IoU0.3': r[f'precision@IoU0.3_conf{CONF}'],
        })

df = pd.DataFrame(rows)
df.set_index(['Dataset', 'Model'], inplace=True)
df.style.format('{:.4f}', subset=df.columns[1:]).highlight_max(axis=0, subset=df.columns[1:])

## 2. Metrics Bar Chart

In [ ]:
dataset_names = list(summary_a.keys())
short_names = ['dataset2_split\ntest', 'cadica_50plus\ntrain', 'arcade\ntest']

metrics_keys = ['mAP@0.25', 'mAP@0.5', 'recall@IoU0.25_conf0.15', 'recall@IoU0.3_conf0.15']
metric_labels = ['mAP@0.25', 'mAP@0.50', 'Recall@IoU≥0.25', 'Recall@IoU≥0.3']

fig, axes = plt.subplots(1, len(metrics_keys), figsize=(20, 5), sharey=False)

for ax, mk, ml in zip(axes, metrics_keys, metric_labels):
    vals_a = [summary_a[d][mk] for d in dataset_names]
    vals_b = [summary_b[d][mk] for d in dataset_names]
    
    x = np.arange(len(dataset_names))
    w = 0.35
    bars_a = ax.bar(x - w/2, vals_a, w, label=MODEL_A, color='#2196F3')
    bars_b = ax.bar(x + w/2, vals_b, w, label=MODEL_B, color='#FF9800')
    
    for bar, v in zip(bars_a, vals_a):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    for bar, v in zip(bars_b, vals_b):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    
    ax.set_xticks(x)
    ax.set_xticklabels(short_names, fontsize=9)
    ax.set_title(ml, fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=7)

fig.suptitle('RF-DETR Model Comparison — Cross-Dataset Evaluation (conf≥0.15)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 3. Helper functions

In [ ]:
def read_yolo_boxes(lbl_path, w, h):
    boxes = []
    if not lbl_path.exists() or lbl_path.stat().st_size == 0:
        return boxes
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5: continue
        cx, cy, bw, bh = map(float, parts[1:5])
        boxes.append(((cx - bw/2)*w, (cy - bh/2)*h, (cx + bw/2)*w, (cy + bh/2)*h))
    return boxes

def draw_rects(ax, boxes, color, lw=2, show_conf=False):
    for b in boxes:
        if show_conf:
            x1, y1, x2, y2, conf = b
        else:
            x1, y1, x2, y2 = b
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, lw=lw, ec=color, fc='none'))
        if show_conf:
            ax.text(x1, y1-3, f'{conf:.2f}', color=color, fontsize=7,
                    bbox=dict(boxstyle='round,pad=0.1', fc='black', alpha=0.6))

def load_preds(eval_dir, name):
    with open(eval_dir / f'preds_{name}.json') as f:
        return json.load(f)

def pred_to_boxes(preds_dict, fname, thresh):
    p = preds_dict.get(fname, {'boxes': [], 'scores': []})
    result = []
    for box, score in zip(p['boxes'], p['scores']):
        if score >= thresh:
            result.append((*box, score))
    return result

## 4. Visualization — dataset2_split test

3 columns: GT | arcade_dataset2_trainval | dataset2_augs

In [ ]:
IMG_DIR = ROOT / 'data' / 'dataset2_split' / 'test' / 'images'
LBL_DIR = ROOT / 'data' / 'dataset2_split' / 'test' / 'labels'
preds_a = load_preds(EVAL_DIR_A, 'dataset2_split_test')
preds_b = load_preds(EVAL_DIR_B, 'dataset2_split_test')

all_imgs = sorted(IMG_DIR.glob('*.jpg'))
imgs_with_gt = [p for p in all_imgs if (LBL_DIR / (p.stem + '.txt')).exists() and (LBL_DIR / (p.stem + '.txt')).stat().st_size > 0]
idx = np.linspace(0, len(imgs_with_gt)-1, 8, dtype=int)
selected = [imgs_with_gt[i] for i in idx]

fig, axes = plt.subplots(len(selected), 3, figsize=(24, 5*len(selected)))
for i, p in enumerate(selected):
    img = np.array(Image.open(p))
    h, w = img.shape[:2]
    gt = read_yolo_boxes(LBL_DIR / (p.stem + '.txt'), w, h)
    det_a = pred_to_boxes(preds_a, p.name, CONF)
    det_b = pred_to_boxes(preds_b, p.name, CONF)
    
    for j, (title, det, color) in enumerate([
        (f'GT ({len(gt)})', None, 'lime'),
        (f'{MODEL_A} ({len(det_a)})', det_a, 'red'),
        (f'{MODEL_B} ({len(det_b)})', det_b, 'cyan'),
    ]):
        ax = axes[i, j]
        ax.imshow(img, cmap='gray'); ax.axis('off')
        ax.set_title(title, fontsize=9)
        if det is None:
            draw_rects(ax, gt, 'lime')
        else:
            draw_rects(ax, gt, 'lime', lw=1)
            draw_rects(ax, det, color, show_conf=True)

fig.suptitle('dataset2_split test — GT (green) vs Predictions', fontsize=14, y=1.001)
plt.tight_layout()
plt.show()

## 5. Visualization — cadica_50plus train (as test)

3 columns: GT | arcade_dataset2_trainval | dataset2_augs

In [ ]:
IMG_DIR = ROOT / 'data' / 'cadica_split_50plus' / 'train' / 'images'
LBL_DIR = ROOT / 'data' / 'cadica_split_50plus' / 'train' / 'labels'
preds_a = load_preds(EVAL_DIR_A, 'cadica_50plus_train')
preds_b = load_preds(EVAL_DIR_B, 'cadica_50plus_train')

all_imgs = sorted(IMG_DIR.glob('*.png'))
imgs_with_gt = [p for p in all_imgs if (LBL_DIR / (p.stem + '.txt')).exists() and (LBL_DIR / (p.stem + '.txt')).stat().st_size > 0]
idx = np.linspace(0, len(imgs_with_gt)-1, 8, dtype=int)
selected = [imgs_with_gt[i] for i in idx]

fig, axes = plt.subplots(len(selected), 3, figsize=(24, 5*len(selected)))
for i, p in enumerate(selected):
    img = np.array(Image.open(p))
    h, w = img.shape[:2]
    gt = read_yolo_boxes(LBL_DIR / (p.stem + '.txt'), w, h)
    det_a = pred_to_boxes(preds_a, p.name, CONF)
    det_b = pred_to_boxes(preds_b, p.name, CONF)
    
    for j, (title, det, color) in enumerate([
        (f'GT ({len(gt)})', None, 'lime'),
        (f'{MODEL_A} ({len(det_a)})', det_a, 'red'),
        (f'{MODEL_B} ({len(det_b)})', det_b, 'cyan'),
    ]):
        ax = axes[i, j]
        ax.imshow(img, cmap='gray'); ax.axis('off')
        ax.set_title(title, fontsize=9)
        if det is None:
            draw_rects(ax, gt, 'lime')
        else:
            draw_rects(ax, gt, 'lime', lw=1)
            draw_rects(ax, det, color, show_conf=True)

fig.suptitle('cadica_50plus train — GT (green) vs Predictions', fontsize=14, y=1.001)
plt.tight_layout()
plt.show()

## 6. Visualization — stenosis_arcade (singlelabel) test

3 columns: GT | arcade_dataset2_trainval | dataset2_augs

In [ ]:
IMG_DIR = ROOT / 'data' / 'stenosis_arcade_singlelabel' / 'test' / 'images'
LBL_DIR = ROOT / 'data' / 'stenosis_arcade_singlelabel' / 'test' / 'labels'
preds_a = load_preds(EVAL_DIR_A, 'stenosis_arcade_singlelabel_test')
preds_b = load_preds(EVAL_DIR_B, 'stenosis_arcade_singlelabel_test')

all_imgs = sorted(IMG_DIR.glob('*.png'))
imgs_with_gt = [p for p in all_imgs if (LBL_DIR / (p.stem + '.txt')).exists() and (LBL_DIR / (p.stem + '.txt')).stat().st_size > 0]
idx = np.linspace(0, len(imgs_with_gt)-1, 8, dtype=int)
selected = [imgs_with_gt[i] for i in idx]

fig, axes = plt.subplots(len(selected), 3, figsize=(24, 5*len(selected)))
for i, p in enumerate(selected):
    img = np.array(Image.open(p))
    h, w = img.shape[:2]
    gt = read_yolo_boxes(LBL_DIR / (p.stem + '.txt'), w, h)
    det_a = pred_to_boxes(preds_a, p.name, CONF)
    det_b = pred_to_boxes(preds_b, p.name, CONF)
    
    for j, (title, det, color) in enumerate([
        (f'GT ({len(gt)})', None, 'lime'),
        (f'{MODEL_A} ({len(det_a)})', det_a, 'red'),
        (f'{MODEL_B} ({len(det_b)})', det_b, 'cyan'),
    ]):
        ax = axes[i, j]
        ax.imshow(img, cmap='gray'); ax.axis('off')
        ax.set_title(title, fontsize=9)
        if det is None:
            draw_rects(ax, gt, 'lime')
        else:
            draw_rects(ax, gt, 'lime', lw=1)
            draw_rects(ax, det, color, show_conf=True)

fig.suptitle('stenosis_arcade (singlelabel) test — GT (green) vs Predictions', fontsize=14, y=1.001)
plt.tight_layout()
plt.show()

## 7. Per-Dataset Detailed Metrics

In [ ]:
for model_name, eval_dir in [(MODEL_A, EVAL_DIR_A), (MODEL_B, EVAL_DIR_B)]:
    print(f"\n{'#'*70}")
    print(f"  MODEL: {model_name}")
    print(f"{'#'*70}")
    for ds in ['dataset2_split_test', 'cadica_50plus_train', 'stenosis_arcade_singlelabel_test']:
        with open(eval_dir / f'eval_{ds}.json') as f:
            r = json.load(f)
        print(f"\n  {'='*55}")
        print(f"  {r['name']}  ({r['num_images']} images, {r['total_gt']} GT boxes)")
        print(f"  {'='*55}")
        for k, v in sorted(r.items()):
            if k in ('name', 'num_images'):
                continue
            if isinstance(v, float):
                print(f"    {k:42s}  {v:.4f}")
            else:
                print(f"    {k:42s}  {v}")